In [35]:
import pandas as pd

In [36]:
# Themes to load
themes = ["Tenure", "Ethnic_group", "Occupation", "Age_structure"]

# Base directories
data_dir_2011 = "../Dataset/Census London 2011/"
data_dir_2021 = "../Dataset/Census London 2021/"

# Merge key
merge_key = "LSOA21CD"

# Dictionary to store each theme's merged data
merged_data = {}

for theme in themes:
    # Construct file paths
    file_2011 = f"{data_dir_2011}{theme}_2011 mapping.csv"
    file_2021 = f"{data_dir_2021}{theme}_2021.csv"

    try:
        # Load data
        df_2011 = pd.read_csv(file_2011).dropna(axis=1, how="all")
        df_2021 = pd.read_csv(file_2021).dropna(axis=1, how="all")

        # Remove 'Borough' column from 2021 data if it exists
        if "Borough" in df_2021.columns:
            df_2021.drop(columns=["Borough"], inplace=True)

        # Add suffixes to all columns except the merge key
        df_2011 = df_2011.rename(columns={col: col + "_2011" for col in df_2011.columns if col != merge_key})
        df_2021 = df_2021.rename(columns={col: col + "_2021" for col in df_2021.columns if col != merge_key})

        # Merge 2011 and 2021 data for this theme
        df_merged = pd.merge(df_2011, df_2021, on=merge_key, how="outer")

        # Store result
        merged_data[theme] = df_merged

        print(f"Successfully loaded and merged: {theme}")

    except Exception as e:
        print(f"Error processing {theme}: {e}")


# Combine all themes into a single DataFrame
df_census = list(merged_data.values())[0]
for df in list(merged_data.values())[1:]:
    df_census = pd.merge(df_census, df, on=merge_key, how="outer")

Successfully loaded and merged: Tenure
Successfully loaded and merged: Ethnic_group
Successfully loaded and merged: Occupation
Successfully loaded and merged: Age_structure


In [37]:
df_property_sales = pd.read_csv("../Output/Number of sales mapping.csv")

In [38]:
df_census = df_census.merge(df_property_sales, left_on='LSOA21CD', right_on='LSOA21CD')

In [39]:
df_census

,LSOA21CD,All households_weighted_2011,Owned_weighted_2011,Owned: Owned outright_weighted_2011,Owned: Owned with a mortgage or loan_weighted_2011,Shared ownership (part owned and part rented)_weighted_2011,Social rented_weighted_2011,Social rented: Rented from council (Local Authority)_weighted_2011,Social rented: Other_weighted_2011,Private rented_weighted_2011,...,Aged 50 to 54 years_2021,Aged 55 to 59 years_2021,Aged 60 to 64 years_2021,Aged 65 to 69 years_2021,Aged 70 to 74 years_2021,Aged 75 to 79 years_2021,Aged 80 to 84 years_2021,Aged 85 years and over_2021,Number of sales 2011_weighted,Number of sales 2021_weighted
0,E01000001,876.0,533.0,355.0,178.0,3.0,41.0,33.0,8.0,264.0,...,89,73,83,119,102,57,57,35,141.0,123.0
1,E01000002,830.0,527.0,314.0,213.0,8.0,48.0,44.0,4.0,219.0,...,122,88,87,76,69,59,43,30,233.0,144.0
2,E01000003,817.0,327.0,184.0,143.0,1.0,295.0,239.0,56.0,177.0,...,155,118,111,86,85,50,31,33,168.0,95.0
3,E01000005,467.0,46.0,24.0,22.0,0.0,312.0,133.0,179.0,101.0,...,87,82,67,35,26,17,14,12,41.0,10.0
4,E01000006,543.0,345.0,136.0,209.0,0.0,18.0,11.0,7.0,178.0,...,121,85,70,66,41,18,17,16,44.0,58.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,E01035718,996.0,502.0,380.0,122.0,4.0,44.0,7.0,37.0,363.0,...,113,142,79,62,82,39,39,40,189.0,113.0
4990,E01035719,528.0,193.0,103.0,90.0,10.0,156.0,95.0,61.0,151.0,...,90,83,61,47,35,31,13,11,75.0,41.0
4991,E01035720,597.0,218.0,117.0,101.0,12.0,176.0,107.0,69.0,170.0,...,118,72,56,33,36,15,18,15,84.0,46.0
4992,E01035721,1296.0,294.0,186.0,108.0,19.0,601.0,272.0,329.0,362.0,...,194,173,137,85,95,67,35,40,100.0,89.0


In [40]:
print(df_census.columns.to_list())

['LSOA21CD', 'All households_weighted_2011', 'Owned_weighted_2011', 'Owned: Owned outright_weighted_2011', 'Owned: Owned with a mortgage or loan_weighted_2011', 'Shared ownership (part owned and part rented)_weighted_2011', 'Social rented_weighted_2011', 'Social rented: Rented from council (Local Authority)_weighted_2011', 'Social rented: Other_weighted_2011', 'Private rented_weighted_2011', 'Private rented: Private landlord or letting agency_weighted_2011', 'Private rented: Other_weighted_2011', 'Living rent free_weighted_2011', 'Total: All households_2021', 'Owned_2021', 'Owned: Owns outright_2021', 'Owned: Owns with a mortgage or loan_2021', 'Shared ownership_2021', 'Shared ownership: Shared ownership_2021', 'Social rented_2021', 'Social rented: Rents from council or Local Authority_2021', 'Social rented: Other social rented_2021', 'Private rented_2021', 'Private rented: Private landlord or letting agency_2021', 'Private rented: Other private rented_2021', 'Lives rent free_2021', 'O

In [41]:
df_census['pct_owner_occupied_2011'] = df_census['Owned_weighted_2011']  / df_census['All households_weighted_2011']
df_census['pct_owner_occupied_2021'] = df_census['Owned_2021']  / df_census['Total: All households_2021']
df_census['delta_owner_occupied'] = df_census['pct_owner_occupied_2021'] - df_census['pct_owner_occupied_2011']

In [42]:
df_census['pct_private_rented_2011'] = df_census['Private rented_weighted_2011']  / df_census['All households_weighted_2011']
df_census['pct_private_rented_2021'] = df_census['Private rented_2021']  / df_census['Total: All households_2021']
df_census['delta_private_rented'] = df_census['pct_private_rented_2021'] - df_census['pct_private_rented_2011']

In [43]:
df_census['pct_social_rented_2011'] = df_census['Social rented_weighted_2011'] / df_census['All households_weighted_2011']
df_census['pct_social_rented_2021'] = df_census['Social rented_2021'] / df_census['Total: All households_2021']
df_census['delta_social_rented'] = df_census['pct_social_rented_2021'] - df_census['pct_social_rented_2011']

In [44]:
df_census['pct_professional_2011'] = (df_census['1. Managers, directors and senior officials_weighted_2011'] + 
                                      df_census['2. Professional occupations_weighted_2011']) / df_census['All categories: Occupation_weighted_2011']
df_census['pct_professional_2021'] = (df_census['1. Managers, directors and senior officials_2021'] + df_census['2. Professional occupations_2021']) / df_census['Total: All usual residents aged 16 years and over in employment the week before the census_2021']
df_census['delta_professional'] = df_census['pct_professional_2021'] - df_census['pct_professional_2011']

In [45]:
df_census['age_20_29_2011'] = (df_census['Age 25 to 29_weighted_2011'] + 
                               df_census['Age 20 to 24_weighted_2011']) / df_census['All usual residents_weighted_2011']
df_census['age_20_29_2021'] = (df_census['Aged 25 to 29 years_2021'] + 
                               df_census['Aged 20 to 24 years_2021']) / df_census['Total: All usual residents_2021']
df_census['delta_age_20_29'] = df_census['age_20_29_2021'] - df_census['age_20_29_2011']

In [46]:
df_census['pct_minority_ethnic_2011'] = (df_census['Mixed/multiple ethnic groups_weighted_2011'] + 
                                         df_census['Asian/Asian British_weighted_2011'] + 
                                         df_census['Black/African/Caribbean/Black British_weighted_2011'] + 
                                         df_census['Other ethnic group_weighted_2011']) / df_census['All usual residents_weighted_2011']
df_census['pct_minority_ethnic_2021'] = (df_census['Mixed or Multiple ethnic groups_2021'] + 
                                         df_census['Asian, Asian British or Asian Welsh_2021'] + 
                                         df_census['Black, Black British, Black Welsh, Caribbean or African_2021'] + 
                                         df_census['Other ethnic group_2021']) / df_census['Total: All usual residents_2021']
df_census['delta_minority_ethnic'] = df_census['pct_minority_ethnic_2021'] - df_census['pct_minority_ethnic_2011'] 

In [47]:
df_census['delta_property_sales'] = df_census['Number of sales 2021_weighted'] - df_census['Number of sales 2011_weighted']

In [48]:
df_census = df_census[['LSOA21CD', 
                'delta_age_20_29', 'delta_property_sales', 'delta_minority_ethnic',
                'pct_professional_2011', 'pct_professional_2021', 'delta_professional', 
                'pct_owner_occupied_2011', 'pct_owner_occupied_2021', 'delta_owner_occupied',
                'delta_social_rented', 'pct_social_rented_2011', 'pct_social_rented_2021',  
                'pct_private_rented_2011', 'pct_private_rented_2021', 'delta_private_rented']]

In [49]:
df_census

,LSOA21CD,delta_age_20_29,delta_property_sales,delta_minority_ethnic,pct_professional_2011,pct_professional_2021,delta_professional,pct_owner_occupied_2011,pct_owner_occupied_2021,delta_owner_occupied,delta_social_rented,pct_social_rented_2011,pct_social_rented_2021,pct_private_rented_2011,pct_private_rented_2021,delta_private_rented
0,E01000001,0.030362,-18.0,0.043830,0.661987,0.688431,0.026444,0.608447,0.574671,-0.033776,-0.020519,0.046804,0.026284,0.301370,0.393070,0.091701
1,E01000002,0.099997,-89.0,0.105080,0.646154,0.723235,0.077081,0.634940,0.524213,-0.110727,-0.025144,0.057831,0.032688,0.263855,0.440678,0.176823
2,E01000003,0.011368,-73.0,0.042489,0.489529,0.580000,0.090471,0.400245,0.372414,-0.027831,-0.063540,0.361077,0.297537,0.216646,0.328079,0.111433
3,E01000005,-0.027058,-31.0,0.128601,0.273109,0.366397,0.093288,0.098501,0.083682,-0.014819,0.028559,0.668094,0.696653,0.216274,0.217573,0.001299
4,E01000006,-0.064914,14.0,-0.000843,0.256082,0.226607,-0.029475,0.635359,0.488246,-0.147113,0.006634,0.033149,0.039783,0.327808,0.464738,0.136929
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4989,E01035718,-0.024713,-76.0,0.014017,0.492778,0.572354,0.079576,0.504016,0.454918,-0.049098,-0.018220,0.044177,0.025956,0.364458,0.512295,0.147837
4990,E01035719,0.045407,-34.0,0.129578,0.516755,0.583824,0.067069,0.365530,0.344392,-0.021139,-0.033211,0.295455,0.262243,0.285985,0.385466,0.099481
4991,E01035720,0.058255,-38.0,0.105360,0.515625,0.485337,-0.030288,0.365159,0.250847,-0.114312,0.117057,0.294807,0.411864,0.284757,0.306780,0.022023
4992,E01035721,-0.021252,-11.0,0.115505,0.366980,0.495814,0.128834,0.226852,0.201224,-0.025628,0.019816,0.463735,0.483550,0.279321,0.306044,0.026723


In [50]:
df_census.to_csv("../Output/data for gentrification classification_.csv", index=False)